# Assignment 7 — Pareto Front Visualisation & Regional Pathways

**Course:** EPA141A Model-Based Decision Making — Delft University of Technology  
**Model:** JUSTICE  

> **Pre-requisite:** Assignment 6 (MOEA Convergence & Reference Set).

---

## Learning Outcomes

After completing this assignment you will be able to:

1. Read and discuss **policy trade-offs** from a parallel-coordinates visualisation.
2. Interpret **regionalised emission control pathways** on a global map.

---

## Background

### Trade-off analysis

A solution is **Pareto-optimal** if you cannot improve one objective without worsening another. The **parallel-coordinates plot** is the standard visualisation for exploring trade-offs across 4+ objectives simultaneously:

- Each vertical axis represents one objective.
- Each line represents one non-dominated policy.
- A **crossing** between two adjacent axes reveals a trade-off: solutions that score well on one objective tend to score poorly on the other.
- Axes are oriented so that all objectives point to the same direction of preference, for instance **up = better** on every axis, making crossings immediately visible.

### Regional emission control rates

The JUSTICE model assigns an **emission control rate (ECR)** to each of the 57 RICE50 world regions. ECR ranges from 0 (no abatement) to 1 (full decarbonisation). The RBF policy computes the ECR for each region based on the current climate state (temperature and its rate of change).

Visualising the ECR at the end of the simulation horizon on a world map reveals **which regions are expected to decarbonise most aggressively** under each policy, and how this varies between policies.

---

## Overview

Starting from the reference set saved in Assignment 6, you will:

1. **Visualise the Pareto front** with a parallel-coordinates plot showing all four objectives.
2. **Map regional ECR pathways** by re-running JUSTICE with four of your selected policies and plotting the end-of-horizon ECR per RICE50 region on a world map.

## Setup — Imports and configuration

The cell below imports all required packages, applies the Python 3.14 compatibility patch for `matplotlib.path.Path`, and loads the cross-seed reference set produced in Assignment 6.

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import os, sys, json, glob, copy
import numpy as np
import pandas as pd
import matplotlib.path as _mpath

def _fixed_path_deepcopy(self, memo):
    cls   = type(self)
    verts = copy.deepcopy(self.vertices, memo)
    codes = copy.deepcopy(self.codes, memo) if self.codes is not None else None
    new   = cls.__new__(cls)
    new.__init__(verts, codes)
    return new

_mpath.Path.__deepcopy__ = _fixed_path_deepcopy

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})


# ── EMA Workbench ─────────────────────────────────────────────────────────────
from ema_workbench.em_framework.optimization import (
    epsilon_nondominated,
    PlatypusProblem,
)
from platypus import Real

# ── Geopandas for maps ───────────────────────────────────────────────────────
import geopandas as gpd

# ── Path setup ────────────────────────────────────────────────────────────────
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))
_PLOTS_DIR  = os.path.join(_NOTEBOOK_DIR, "plots")
os.makedirs(_PLOTS_DIR, exist_ok=True)

# ── Objective metadata ────────────────────────────────────────────────────────
OBJECTIVE_COLS  = ["welfare", "fraction_above_threshold",
                   "welfare_loss_damage", "welfare_loss_abatement"]
MAXIMIZE_COLS   = ["welfare_loss_damage", "welfare_loss_abatement"]
MINIMIZE_COLS   = ["welfare", "fraction_above_threshold"]
OBJECTIVE_LABELS = {
    "welfare":                  "Welfare loss\n(MINIMIZE)",
    "fraction_above_threshold": "Fraction above\n2°C in 2100\n(MINIMIZE)",
    "welfare_loss_damage":      "Welfare loss\nfrom damage\n(MAXIMIZE)",
    "welfare_loss_abatement":   "Welfare loss\nfrom abatement\n(MAXIMIZE)",
}

print(f"JUSTICE root : {_JUSTICE_ROOT}")
print(f"Results root : {RESULTS_ROOT}")
print(f"DEAP hv      : OK")
print(f"geopandas    : {gpd.__version__}")
print("Matplotlib deepcopy patch applied.")

## Load the Reference Set

This cell loads the Pareto-optimal reference set you generated in Assignment 6.

> **Before running:** replace `<name_of_your_reference_set>` with the actual filename of your saved reference set.


In [ ]:
# ── Load grand reference set from Assignment 6 ────────────────────────────────

ref_path = os.path.join(RESULTS_ROOT, "<name_of_your_reference_set>.csv")

if not os.path.exists(ref_path):
    raise FileNotFoundError(
        f"Reference set not found: {ref_path}\n"
        "Run Assignment 6 (Steps 1-3) first to build and save the reference set."
    )

ref_set = pd.read_csv(ref_path)
ref_set.columns = [c.replace(" ", "_") for c in ref_set.columns]
ref_set = ref_set[ref_set["welfare"] < 1e5].reset_index(drop=True)


print(f"Reference set loaded: {len(ref_set)} solutions × {len(ref_set.columns)} columns")
print(f"\nObjective statistics:")
print(ref_set[OBJECTIVE_COLS].describe().round(3).to_string())

---

## Step 1 — Pareto Front Visualisation & Trade-off Discussion


### Parallel-Coordinates Plot

Create a **parallel-coordinates plot** of your reference set that lets you
visually compare all solutions across every objective at once.

**What to produce**

- One line per solution in the reference set; every axis represents one objective.
- All axes should point to the same direction of preference for instance..if **upward = better**, you would normalise your data accordingly
  (invert minimisation objectives after scaling to [0, 1]).
- Colour the lines by one objective of your choice to reveal trade-off patterns.
- Highlight four solutions, based on your criteria,  using thick, distinctly coloured lines and a legend.
- Add a colour bar, labelled axes, and a title.

**Useful resources**

- [`ema_workbench` built-in parallel coordinates](https://emaworkbench.readthedocs.io/en/latest/ema_documentation/analysis/parcoords.html)
  — the quickest route; wraps matplotlib and handles normalisation for you.

- [Pandas `DataFrame.plot`](https://pandas.pydata.org/docs/reference/api/pandas.plotting.parallel_coordinates.html)



In [ ]:
#YOUR CODE HERE

### Trade-off Discussion

Interpret your parallel-coordinates plot. Your answer should cover:

1. **Pairwise trade-offs** — for each pair of adjacent axes, does a crossing appear? What does it mean for the relationship between those two objectives?
2. **Dominant trade-off** — which pair shows the sharpest conflict? 
3. **Synthesis** — in 2 sentences, what is the fundamental tension in this problem as revealed by the Pareto front?


---

## Step 2 — Regional Emission Control Rate (ECR) Pathways

During the optimisation, each Pareto-optimal solution was stored as a set of
**RBF parameters** (centers, radii, weights) plus its four objective values.
The parameters encode *how* the policy responds to the climate state — but the
full regional detail (which region gets how much emission control, and when) was
never saved.

The following cell takes care of that.. 

**1. Import tools**
Loads all the components needed to run the JUSTICE model — the model itself, the RBF policy structure, time settings, region data, and constraint logic.

**2. Load your config file**
Opens your config file (make sure you specify your config file in the cell) and reads the model settings (start/end year, timestep size, scenario index, etc.) so everything is consistent with how the optimisation was originally run.

**3. Set up constants**
Calculates how many timesteps and regions the model has, and stores the temperature bounds used for scaling later.

**4. Define a helper function: `run_policy_ecr(policy_row)`**
When called with a single policy (one row from the reference set), this function:

- **Rebuilds the RBF** — extracts the centers, radii, and weights stored in that row, recreating the decision rule that was optimised.
- **Sets up a constraint** — limits how fast emission control can grow each year so the model behaves realistically. 
- **Initialises JUSTICE** — sets up the model with the same scenario and assumptions used during optimisation.
- **Runs the model one timestep at a time** — at each step it applies the constraint, advances the model, reads the resulting global temperature, scales it to [0, 1], and feeds it back into the RBF to decide the next timestep's emission control rate. The RBF is essentially a rule: given the climate state right now, apply this much control to each region. It's static in its parameters, but its output varies because the climate state it reads changes at every step. You're re-running the policy forward through time to recover the full regional detail that wasn't saved during optimisation
- **Returns** the mean emission control rate across regions over time, plus the full model output.

**5. Select your policies**
Filters the reference set to the objective columns, then asks you to pick one policy per objective using for instance `.idxmin()` or `.idxmax()`, but  the idea is to use the index pointing to your selected policies.

**6. Print a summary**
Prints each selected policy's objective values so you can verify your selection before running.

**7. Run the model**
Calls `run_policy_ecr()` once per selected policy to get its full emission control trajectory.


In [ ]:
# ── JUSTICE imports ───────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

from justice.model import JUSTICE
from justice.util.data_loader import DataLoader
from justice.util.enumerations import (
    Abatement, DamageFunction, Economy, WelfareFunction
)
from justice.util.emission_control_constraint import EmissionControlConstraint
from justice.util.model_time import TimeHorizon
from solvers.emodps.rbf import RBF

# ── Model constants ─────────────────────────

#TODO:
#Before you begin: Update the path in the cell below 
#to point to the config file you created previously. 
#Make sure it reflects the scenario and settings you have been working
#with throughout. 

with open(os.path.join(_NOTEBOOK_DIR, "../config/<your_config_file>.json")) as fh:
    _cfg = json.load(fh)

_time_horizon = TimeHorizon(
    start_year            = _cfg["start_year"],
    end_year              = _cfg["end_year"],
    data_timestep         = _cfg["data_timestep"],
    timestep              = _cfg["timestep"],
)
N_TIMESTEPS    = len(_time_horizon.model_time_horizon)
N_REGIONS      = len(DataLoader().REGION_LIST)
REGION_LIST    = list(DataLoader().REGION_LIST)
N_INPUTS_RBF   = _cfg["n_inputs"]
N_RBFS         = _cfg["n_inputs"] + 2
SCENARIO       = _cfg["reference_ssp_rcp_scenario_index"]
EC_START_TS    = _time_horizon.year_to_timestep(
    year=_cfg["emission_control_start_year"],
    timestep=_cfg["timestep"],
)
_MAX_TEMP, _MIN_TEMP = 16.0, 0.0
_MAX_DIFF, _MIN_DIFF = 2.0,  0.0

# ── Helper: run JUSTICE for one policy row ────────────────────────────────────
def run_policy_ecr(policy_row, n_ensemble=1):
    rbf = RBF(n_rbfs=N_RBFS, n_inputs=N_INPUTS_RBF, n_outputs=N_REGIONS)
    c_shape, r_shape, w_shape = rbf.get_shape()

    centers = np.array([policy_row[f"center_{i}"] for i in range(c_shape[0])])
    radii   = np.array([policy_row[f"radii_{i}"]  for i in range(r_shape[0])])
    weights = np.array([policy_row[f"weights_{i}"] for i in range(w_shape[0])])
    rbf.set_decision_vars(np.concatenate([centers, radii, weights]))

    constraint = EmissionControlConstraint(
        max_annual_growth_rate=0.04,
        emission_control_start_timestep=EC_START_TS,
        min_emission_control_rate=0.01,
    )

    ensemble_indices = list(np.linspace(1, 1000, max(n_ensemble, 10), dtype=int))[:n_ensemble]

    model = JUSTICE(
        scenario=SCENARIO,
        climate_ensembles=ensemble_indices,
        economy_type=Economy.NEOCLASSICAL,
        damage_function_type=DamageFunction.KALKUHL,
        abatement_type=Abatement.ENERDATA,
        social_welfare_function_type=WelfareFunction.UTILITARIAN.value[0],
    )
    no_ens = model.no_of_ensembles

    ecr             = np.zeros((N_REGIONS, N_TIMESTEPS, no_ens))
    constrained_ecr = np.zeros_like(ecr)
    prev_temp = 0.0
    diff      = 0.0

    for t in range(N_TIMESTEPS):
        constrained_ecr[:, t, :] = constraint.constrain_emission_control_rate(
            ecr[:, t, :], t, allow_fallback=False
        )
        model.stepwise_run(
            emission_control_rate=constrained_ecr[:, t, :],
            timestep=t,
            endogenous_savings_rate=True,
        )
        data = model.stepwise_evaluate(timestep=t)
        temp = data["global_temperature"][t, :]

        if t % 5 == 0:
            diff      = temp - prev_temp
            prev_temp = temp

        scaled_temp = (temp - _MIN_TEMP) / (_MAX_TEMP - _MIN_TEMP)
        scaled_diff = (diff - _MIN_DIFF) / (_MAX_DIFF - _MIN_DIFF)

        if t < N_TIMESTEPS - 1:
            ecr[:, t + 1, :] = rbf.apply_rbfs(np.array([scaled_temp, scaled_diff]))

    datasets = model.evaluate()
    ecr_mean = constrained_ecr.mean(axis=2)
    return ecr_mean, datasets


# ── TODO: Select your  policies ────────────────────────────────────
# Use the same policies you highlighted in your parallel-coordinates plot.
# Replace each line below with the appropriate indeces. 

obj_ref = ref_set[OBJECTIVE_COLS]

idx_1 = obj_ref["<column>"].idxmin()   # example, this would be the row index of your first selected policy, and is looking at the lowest value within that objective column

#add the rest of your objectives...

anchors = {
    "Policy 1 — <your label>": idx_1,
#add the rest of your objectives...
}

print("Selected policies from Pareto reference set:\n")
for label, idx in anchors.items():
    print(f"  {label}")
    print(obj_ref.loc[idx].round(3).to_string(index=True))
    print()

# ── Run JUSTICE for each anchor ───────────────────────────────────────────────
print("Running JUSTICE model for all selected policies ...")
ecr_1, _ = run_policy_ecr(ref_set.loc[idx_1], n_ensemble=1)
# complete the rest of the policies using the example above. 
print("Done.")

### Preparing the Geographic Data

Run the cell below almost as-is (just update the `ecr_end` keys to match your selected policy labels).

**What it does:**

**1. Build a country: RICE50 lookup table**
The JUSTICE model groups the world into 57 aggregate regions (RICE50) rather than ~180 individual countries. To draw a world map, we need to go the other way — start from country shapes and colour each country according to the RICE50 region it belongs to. This step builds that link by reading a lookup table that says, for example, "Angola, Benin, Botswana, ... all belong to region rsaf."

The lookup uses each country's standard 3-letter ISO code (e.g. FRA for France). However, the world map shapefile we use here stores a few countries with the placeholder code '-99' instead of their real ISO code — so the lookup fails for them. The fallback simply matches those countries by name instead.

**2. Take an end-of-horizon snapshot**
From the full ECR time series computed in the previous cell, it extracts only
the **final timestep** — one ECR value per region per selected policy. This single
number summarises how aggressively each region is controlling emissions at the
end of the simulation under that policy.

**3. Attach ECR values to a world map**
It loads a world map (one shape per country), assigns each country to its RICE50 region, and attaches the ECR value to it. Countries that belong to the same RICE50 region are then merged into one shape, giving 57 regional polygons ready to be coloured in the next cell.

The output `regions_gdf` is a GeoDataFrame with 57 rows, ready to plot.


In [ ]:
# ── Map world countries to RICE50 regions via ISO-3166 codes ─────────────────
import importlib.util, pathlib, json as _json

_rice50_dict_path = os.path.join(_JUSTICE_ROOT, "data", "input", "rice50_regions_dict.json")
with open(_rice50_dict_path) as _f:
    _rice50_dict = _json.load(_f)

iso_to_rice50 = {
    iso: region
    for region, isos in _rice50_dict.items()
    for iso in isos
}

_name_fallback = {
    "France":     "fra",
    "Norway":     "nor",
    "Kosovo":     "oeu",
    "N. Cyprus":  "tur",
    "Somaliland": "rsaf",
}

# ── Extract end-of-horizon ECR snapshot ───────────────────────────────────────
t_end     = ecr_1.shape[1] - 1
snap_year = int(_time_horizon.model_time_horizon[t_end])
print(f"End-of-horizon snapshot: timestep {t_end} = year {snap_year}")

def _snap_end(arr):
    return {REGION_LIST[i]: arr[i, t_end] for i in range(N_REGIONS)}

# TODO: replace the keys with your own policy labels (must match what you used above)
ecr_end = {
    "Policy 1": _snap_end(ecr_1),
#complete the function
}

# ── Load world shapefile and assign regions ───────────────────────────────────
_pyogrio_path = pathlib.Path(importlib.util.find_spec("pyogrio").origin).parent
_ne_shp = str(_pyogrio_path / "tests" / "fixtures" / "naturalearth_lowres" / "naturalearth_lowres.shp")
world = gpd.read_file(_ne_shp)

world["rice50"] = world["iso_a3"].map(iso_to_rice50)
mask_missing = world["rice50"].isna()
world.loc[mask_missing, "rice50"] = world.loc[mask_missing, "name"].map(_name_fallback)

for policy, region_ecr in ecr_end.items():
    world[f"ecr_{policy}"] = world["rice50"].map(region_ecr)

n_mapped = world["rice50"].notna().sum()
print(f"Countries mapped to RICE50 regions: {n_mapped}/{len(world)}")

# ── Dissolve into 57 RICE50 region polygons ───────────────────────────────────
ecr_cols_map = [f"ecr_{p}" for p in ecr_end.keys()]

regions_gdf = (
    world[world["rice50"].notna()]
    .dissolve(by="rice50", aggfunc="first")
    .reset_index()
)[["rice50", "geometry"] + ecr_cols_map]

import geopandas as _gpd
if not isinstance(regions_gdf, _gpd.GeoDataFrame):
    regions_gdf = _gpd.GeoDataFrame(regions_gdf, geometry="geometry", crs=world.crs)

print(f"Dissolved to {len(regions_gdf)} RICE50 regions")


### Plotting the Regional Emission Control Rate Maps

This following cell uses a helper code that draws a 2×2 world map showing how each of
the selected four policies distributes emission control across the 57 RICE50 regions
at the end of the simulation. You can run it as is, or replace it with your
own visualisation, the goal is simply to see how emission control rates vary
geographically across regions for each policy.


Here's what the cell does, step by step:

**1. Define what to plot**
Builds a list of panels — one per selected policy.

**2. Set a shared colour scale**
Finds the minimum and maximum ECR values across all panels so every map uses the same color scale to make them directly comparable.

**3. Create a 2×2 grid of maps**
Sets up a figure with four subplots, one per anchor policy.

**4. Draw each map**
For each panel:
- Draws the world in light grey as a background
- Colors each RICE50 region according to its ECR value at the snapshot year, using a yellow–orange–red scale (low → high).


**What to look for**

Which regions carry the heaviest mitigation burden under each policy?


In [ ]:
# ── World map — end-of-horizon ECR snapshot (2×2 grid, one panel per policy) ──
panel_specs = [
    (f"ecr_{label}", f"{label}\nECR snapshot ({snap_year})")
    for label in ecr_end.keys()
]

all_vals = pd.concat([regions_gdf[col].dropna() for col, _ in panel_specs])
vmin_all, vmax_all = all_vals.min(), all_vals.max()

_cmap = plt.cm.YlOrRd
_norm = mcolors.Normalize(vmin=vmin_all, vmax=vmax_all)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes_flat = axes.flatten()

for ax, (col, title) in zip(axes_flat, panel_specs):
    world.plot(ax=ax, color="0.88")
    world.boundary.plot(ax=ax, color="0.7", linewidth=0.2)

    colors = [
        _cmap(_norm(v)) if pd.notna(v) else "0.88"
        for v in regions_gdf[col]
    ]
    regions_gdf.plot(ax=ax, color=colors)
    regions_gdf.boundary.plot(ax=ax, color="0.4", linewidth=0.5)

    sm = plt.cm.ScalarMappable(cmap=_cmap, norm=_norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="horizontal",
                        pad=0.02, shrink=0.75, aspect=35)
    cbar.set_label("Emission Control Rate (0–1)", fontsize=8)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=5)
    ax.set_axis_off()

# TODO: update the title if needed
fig.suptitle(
    f"Regional ECR at End of Simulation Horizon ({snap_year}) — JUSTICE RICE50 Model\n"
    "57 aggregate regions; shared colour scale across panels",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig(os.path.join(_PLOTS_DIR, "ecr_regional_map.png"), dpi=150, bbox_inches="tight")
plt.show()
# print("Figure saved: ecr_regional_map.png")




## Reflection Questions

**1. Trade-offs on the parallel-coordinates plot.** Identify the two axes that show the clearest crossing pattern (i.e., the strongest trade-off). Explain in 2–3 sentences why this trade-off exists.

**2. Emission control rate map.** Under your selected policies, which regions show the largest difference in ECR between the two? 

**3. Actor perspective.** Based on the parallel-coordinates plot and the regional ECR map, which policy from the reference set would you recommend to your actor? 
